# Laboratorio 4: demostración en vivo por voz (para el video)

Este notebook **solo carga lo necesario para la demo** y no repite la evaluación: los resultados completos están en `Laboratorio4_Asistente_Voz.ipynb` (ya ejecutado). Usa el mismo código (`lab4/`) y los mismos datos e índices guardados en Drive.

**Antes de empezar**
1. `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`.
2. Permitir el micrófono: candado a la izquierda de la URL → *Micrófono* → *Permitir*.
3. Usar audífonos (si suena por parlantes, el micrófono capta la voz del asistente).
4. `Entorno de ejecución → Ejecutar todas`. La carga tarda unos 20–25 min (la mayor parte es descargar Qwen3-8B). La última celda **empieza a grabar** cuando llega: tener el guion a mano.

**En la celda de la demo, por turno:** aparece el botón rojo *⏹ Detener grabación* (ya está grabando) → decir la frase → *Detener* → revisar la transcripción (Enter para aceptar o escribir la corrección) → esperar la respuesta, que suena con la voz clonada. El siguiente turno empieza a grabar enseguida: esperar a que termine la respuesta antes de hablar. Escribir `fin` para terminar.

## 1. Entorno

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "coqui-tts[codec]>=0.28" six "sentence-transformers>=4.0" "transformers>=4.57" "bitsandbytes>=0.45" accelerate tabulate
!apt-get -qq install -y ffmpeg > /dev/null
# Si pip actualizó torch/transformers ya importados, reinicie la sesión (Entorno de ejecución → Reiniciar) y continúe desde la celda siguiente.

In [ ]:
import os, sys, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

SOL_DIR = Path('/content/drive/MyDrive/master/deeplearning/laboratorio04/solucion')   # ← ajustar si se usa otra ruta
WORK_DIR = Path('/content/drive/MyDrive/master/deeplearning/laboratorio04/work')
os.environ['LAB4_BASE'] = str(WORK_DIR)
os.environ['COQUI_TOS_AGREED'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(SOL_DIR))

import torch, numpy as np, pandas as pd
from lab4 import config as C
C.ensure_dirs()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "¡sin GPU!")

## 2. Catálogo, buscador y tienda (desde lo guardado en Drive)

In [ ]:
from lab4 import data
data.download_esci()                       # ya está en Drive: no vuelve a descargar
catalog, queries, qrels = data.prepare_data()
print("Catálogo:", catalog.shape)

In [ ]:
from lab4.retrieval import BM25Index, DenseIndex, Reranker, HybridSearcher
from lab4.pipeline import ModelRegistry, gpu_mem, VoiceAssistant
reg = ModelRegistry()
bm25 = BM25Index(C.RETRIEVAL.bm25_k1, C.RETRIEVAL.bm25_b).fit(catalog.title.tolist(), catalog.passage.tolist())
dense = reg.load("embedder", lambda: DenseIndex())
dense.fit(catalog.passage.tolist(), cache_path=C.INDEX_DIR / "e5_base_passages.npy")   # embeddings en caché
reranker = reg.load("reranker", lambda: Reranker())
searcher = HybridSearcher(catalog, bm25, dense, reranker, C.RETRIEVAL)
for h in searcher.search("botellas de agua reutilizables", k=3):
    print(f"{h.score:7.3f}  {catalog.loc[h.idx, 'title'][:80]}")

In [ ]:
from lab4.store import Store
store = Store(C.STORE_DIR / "store.db")
store.reset(catalog.product_id.tolist(), seed=C.SEED)   # misma semilla que la evaluación
store.snapshot("base")
print("Tienda lista:", len(catalog), "productos")

## 3. Modelos: LLM, ASR y TTS con la voz clonada

In [ ]:
from lab4.llm import LocalLLM
from lab4.agent import ShoppingAgent
try:
    llm = reg.load("llm", lambda: LocalLLM(C.MODELS.llm))
except Exception as e:
    print("Fallo cargando", C.MODELS.llm, "→", e, "\nSe usa", C.MODELS.llm_fallback)
    llm = reg.load("llm", lambda: LocalLLM(C.MODELS.llm_fallback))
print(gpu_mem())

In [ ]:
from lab4.speech import record, ASR, VoiceTTS
from IPython.display import Audio, display
REF_WAV = C.AUDIO_DIR / "referencia.wav"
assert REF_WAV.exists(), f"Falta la grabación de referencia en {REF_WAV}"
asr = reg.load("asr", lambda: ASR())
tts = reg.load("tts", lambda: VoiceTTS(REF_WAV))
display(pd.DataFrame(reg.load_report))
print(gpu_mem())

## 4. Demostración en vivo

**Guion sugerido** (diapositiva 11 de la presentación):

| Turno | Qué decir |
|---|---|
| 1 | «Quiero dos botellas de agua reutilizables, mi presupuesto es de cuarenta euros.» |
| 2 | «Me quedo con la primera, prepárame la cotización.» |
| 3 | «Sí, confirmo el pedido.» |
| 4 | «Necesito un cargador.» |
| 5 | «Es para un móvil con USB C, solo uno, el primero que muestres. Cotiza.» |
| 6 | «Sí, confirmo.» → luego escribir `fin` |

Para volver a empezar desde cero, ejecutar otra vez esta celda: la tienda se restaura sola.

In [ ]:
store.restore("base")
live_agent = ShoppingAgent(llm, store, searcher, catalog, session_id="live", trace=True)
va = VoiceAssistant(asr, live_agent, tts)
for k in range(1, 20):
    wav, t_end = record(C.AUDIO_DIR / f"live_t{k}.wav", max_seconds=25, label=f"turno {k}")
    hyp, asr_s = asr.transcribe(wav)
    print(f"📝 Transcripción ({asr_s:.2f} s):", hyp)
    fix = input("Enter para aceptar, texto para corregir, 'fin' para terminar: ").strip()
    if fix.lower() == "fin":
        break
    # La espera de revisión humana se excluye; la latencia = ASR + agente + TTS
    r = va.turn(wav, correction=fix or None, tag=f"live_t{k}", play=True, asr_result=(hyp, asr_s))
    print("🤖", r["response"])
    print("⏱️", {k2: round(v2, 2) for k2, v2 in r["latency"].items()})
print(live_agent.state.to_prompt(live_agent.tools._cart_view()[0]))
print(store.orders("live"))